In [1]:
from pathlib import Path
import sys

from matplotlib.pyplot import title

ROOT = Path.cwd().parent
sys.path.insert(0, str(ROOT))
sys.path.insert(0, str(ROOT / "services"))

# print(Path.cwd(), ROOT)

In [2]:
# import yfinance as yf
#
# gme = yf.download('GME', start='2020-06-01', end='2021-12-31', multi_level_index=False)
# gme

In [3]:
# gme.to_csv(ROOT / 'data' / 'series' / 'GME.csv')

In [4]:
import polars as pl

df = pl.read_csv(ROOT / 'data' / 'series' / 'GME.csv', try_parse_dates=True)
print(df)

shape: (401, 6)
┌────────────┬───────────┬───────────┬───────────┬───────────┬──────────┐
│ Date       ┆ Close     ┆ High      ┆ Low       ┆ Open      ┆ Volume   │
│ ---        ┆ ---       ┆ ---       ┆ ---       ┆ ---       ┆ ---      │
│ date       ┆ f64       ┆ f64       ┆ f64       ┆ f64       ┆ i64      │
╞════════════╪═══════════╪═══════════╪═══════════╪═══════════╪══════════╡
│ 2020-06-01 ┆ 1.0325    ┆ 1.09      ┆ 1.005     ┆ 1.03      ┆ 10446400 │
│ 2020-06-02 ┆ 1.045     ┆ 1.0775    ┆ 1.015     ┆ 1.0675    ┆ 9477600  │
│ 2020-06-03 ┆ 1.11      ┆ 1.13      ┆ 1.05      ┆ 1.06      ┆ 12149600 │
│ 2020-06-04 ┆ 1.1175    ┆ 1.18      ┆ 1.0925    ┆ 1.11      ┆ 14057200 │
│ 2020-06-05 ┆ 1.035     ┆ 1.1025    ┆ 1.0225    ┆ 1.0725    ┆ 25097600 │
│ …          ┆ …         ┆ …         ┆ …         ┆ …         ┆ …        │
│ 2021-12-23 ┆ 38.035    ┆ 38.75     ┆ 36.505001 ┆ 38.5      ┆ 4222000  │
│ 2021-12-27 ┆ 37.077499 ┆ 38.154999 ┆ 35.0      ┆ 38.0      ┆ 6454400  │
│ 2021-12-28 ┆ 36.6150

In [5]:
import altair as alt
import datetime as dt

df = df.filter(
    pl.col('Date').is_between(dt.date(2021, 1, 1), dt.date(2021, 2, 28))
)

price = alt.Chart(df).mark_line().encode(
    x="Date:T",
    y="Close:Q",
).properties(
    width=1200, height=400,
    title=alt.Title("GME", subtitle="Jan & Feb 2021"),
)

rect1 = alt.Chart().mark_rect(color="green", opacity=0.5).encode(
    x=alt.datum(alt.DateTime(year=2021, month=1, date=17)),
    x2=alt.datum(alt.DateTime(year=2021, month=1, date=27)),
)

txt1 = alt.Chart().mark_text(text='short squeeze', align='center').encode(
    x=alt.datum(alt.DateTime(year=2021, month=1, date=22)),
    y=alt.datum(70)
)

rect2 = alt.Chart().mark_rect(color="red", opacity=0.5).encode(
    x=alt.datum(alt.DateTime(year=2021, month=1, date=27)),
    x2=alt.datum(alt.DateTime(year=2021, month=2, date=5)),
)

txt2 = alt.Chart().mark_text(text='"SELL" button', align='center').encode(
    x=alt.datum(alt.DateTime(year=2021, month=1, date=30)),
    y=alt.datum(35)
)

price + rect1 + txt1 + rect2 + txt2

alt.LayerChart(...)

In [6]:
# need to get historical options data
# alphavantage can provide in batches of 25days at a time, I can query some now, then query more next day

from dotenv import load_dotenv
import os

load_dotenv()
KEY = os.environ.get("ALPHAVANTAGE_API_KEY")
# print(KEY)

In [7]:
import requests

def get_GME_historical_options(date: str):
    r = requests.get("https://www.alphavantage.co/query", params={
        "function": "HISTORICAL_OPTIONS",
        "symbol": "GME",
        "date": date,  # "YYYY-MM-DD"
        "apikey": KEY,
    })

    return r.json()

gme_opt = get_GME_historical_options("2021-01-01")
print(gme_opt)

{'Information': 'Thank you for using Alpha Vantage! This is a premium endpoint. You may subscribe to any of the premium plans at https://www.alphavantage.co/premium/ to instantly unlock all premium endpoints'}


In [8]:
# ruling out analysis of GME for this study